In [4]:
import sys
from pathlib import Path
raiz = Path.cwd()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
from src.cuantizacion import (cuantizar_dataframe, reconstruir_dataframe,
                              comparar_estadisticas, tabla_resultados)
DIR_TABLAS = raiz / "resultados" / "actividad3" / "tablas"
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

In [5]:
df = pd.read_parquet(raiz / "data" / "dataset_1M.parquet")
df = df.sample(200_000, random_state=0).reset_index(drop=True)
COLS = ["edad", "ingresos", "altura"]
otras = [c for c in df.columns if c not in COLS]
print(df.dtypes)
print("columnas sin cuantizar:", otras)

edad           float64
ingresos       float64
altura         float64
experiencia      int32
categoria       object
dtype: object
columnas sin cuantizar: ['experiencia', 'categoria']


In [6]:
copia = df.copy()
df_cod, modelos = cuantizar_dataframe(df, COLS, "cuantiles", 10)

pd.testing.assert_frame_equal(df, copia)                  # original intacto
assert df_cod.shape == df.shape and list(df_cod.columns) == list(df.columns)
pd.testing.assert_frame_equal(df_cod[otras], df[otras])   # no cuantizadas idénticas
print(df_cod[COLS].dtypes)
df_cod.head()

edad        int16
ingresos    int16
altura      int16
dtype: object


,edad,ingresos,altura,experiencia,categoria
0,2,1,4,15,C
1,1,3,6,14,A
2,3,0,8,18,B
3,9,8,9,6,A
4,0,2,7,14,A


In [7]:
df_rec = reconstruir_dataframe(df_cod, modelos, "media")
est = comparar_estadisticas(df[COLS], df_rec[COLS])
display(est["resumen"].round(4))
display(est["dif_correlacion"].round(5))
print("diferencia máxima de correlación:", est["dif_correlacion_max"])

,columna,dif_media,dif_std,dif_q05,dif_q25,dif_q50,dif_q75,dif_q95
0,edad,-0.0,-0.2486,-1.3122,-0.044,0.0004,0.0461,1.3125
1,ingresos,0.0,-723.6535,-330.7431,-12.979,57.5182,81.3846,4084.7656
2,altura,-0.0,-0.2107,-1.1281,-0.037,-0.0042,0.0119,1.1614


,edad,ingresos,altura
edad,0.00000,0.00179,0.00034
ingresos,0.00179,0.00000,-0.00024
altura,0.00034,-0.00024,0.00000


diferencia máxima de correlación: 0.0017882297482646275


In [8]:
tabla = tabla_resultados(df, COLS)
tabla.to_csv(DIR_TABLAS / "tabla_resultados_demo.csv", index=False)
print(tabla.shape)          # (72, 20): 3 columnas x 2 estrategias x 3 representantes x 4 bins
tabla.head()

(72, 19)


,columna,estrategia,representante,bins_solicitados,bins_reales,mae,rmse,error_max,dif_media,dif_std,dif_q05,dif_q25,dif_q50,dif_q75,dif_q95,dif_corr_max,t_ajuste_s,t_transformar_s,t_reconstruir_s
0,edad,uniforme,punto_medio,5,5,5.119875,5.913285,10.242299,-1.765804e-02,1.365097,-4.618994,4.158618,-3.936699,8.486650,-3.220165,0.002638,0.007544,0.002907,0.000912
1,ingresos,uniforme,punto_medio,5,5,7471.381943,8972.025109,19507.841814,1.168921e+03,-509.565372,12290.102018,6195.895554,-173.204297,-9021.668922,10655.281181,0.002638,0.006673,0.002072,0.000783
2,altura,uniforme,punto_medio,5,5,4.805506,5.553120,9.639635,-1.570584e-02,1.359568,-4.439600,5.128566,-1.595953,-8.319780,1.307057,0.002638,0.007645,0.003313,0.000966
3,edad,uniforme,media,5,5,4.453351,5.306251,17.086923,-1.563194e-13,-1.235373,0.015292,5.033110,-3.062207,5.104245,-6.602570,0.003035,0.007544,0.002907,0.001013
4,ingresos,uniforme,media,5,5,7017.971284,8489.772351,29471.834069,-8.367351e-11,-3047.631148,12025.047913,5930.841450,-438.258402,-9286.723027,1632.396375,0.003035,0.006673,0.002072,0.000713


In [9]:
tabla.pivot_table(index=["columna", "estrategia", "bins_solicitados"],
                  columns="representante", values="mae").round(4)

representante                             media    mediana  punto_medio
columna  estrategia bins_solicitados                                   
altura   cuantiles  5                    2.3060     2.2577       6.5898
                    10                   1.2830     1.2579       3.2413
                    25                   0.5725     0.5636       1.2555
                    50                   0.3074     0.3028       0.6136
         uniforme   5                    4.0771     4.0433       4.8055
                    10                   2.2942     2.2831       2.4108
                    25                   0.9583     0.9572       0.9662
                    50                   0.4802     0.4801       0.4810
edad     cuantiles  5                    2.7679     2.7120       6.6303
                    10                   1.5355     1.5076       3.2563
                    25                   0.6879     0.6767       1.2620
                    50                   0.3685     0.3631       0.6129
         uniforme   5                    4.4534     4.4151       5.1199
                    10                   2.4656     2.4558       2.5629
                    25                   1.0179     1.0171       1.0243
                    50                   0.5122     0.5120       0.5131
ingresos cuantiles  5                 3155.9280  3036.7122   15553.5952
                    10                1827.9875  1764.0010    7561.0066
                    25                 869.3060   841.7828    2898.9729
                    50                 488.8789   474.5113    1398.1330
         uniforme   5                 7017.9713  6953.2905    7471.3819
                    10                3952.6971  3921.1883    4684.5232
                    25                1865.3151  1861.1841    1956.3961
                    50                 962.6688   961.5699     976.1790